# 03 · Visualize

讀 `data/processed/{village|grid}_to_nearest_library_{driving|walking}.csv`
+ 對應的幾何（村里 geojson 或 grid_*m.geojson），輸出多張地圖。

每組 (UNIT, METRIC) 產出靜態 PNG 與互動 HTML：

- `tainan_library_drive_time_{village|grid}_*` — 行車時間
- `tainan_library_drive_dist_{village|grid}_*` — 行車距離
- `tainan_library_walk_time_{village|grid}_*` — 步行時間

不存在的 CSV 會 skip。


In [ ]:
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from lib.colors import DRIVING, WALKING, DISTANCE

RAW_DIR = ROOT / "data" / "raw"
PROC_DIR = ROOT / "data" / "processed"
OUTPUT_DIR = ROOT / "output" / "maps"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    plt.rcParams["font.sans-serif"] = ["PingFang TC", "Heiti TC", "Arial Unicode MS"]
    plt.rcParams["axes.unicode_minus"] = False
except Exception:
    pass

# Load geometry sources
villages = gpd.read_file(RAW_DIR / "tainan_villages.geojson")
villages = villages.rename(columns={"village_id": "unit_id"})
libs = pd.read_csv(RAW_DIR / "tainan_libraries.csv")
libs_gdf = gpd.GeoDataFrame(libs, geometry=gpd.points_from_xy(libs.lon, libs.lat), crs="EPSG:4326")

def find_grid_geojson():
    """Pick the most-recently modified grid_*m.geojson cache (if any)."""
    cands = sorted(PROC_DIR.glob("grid_*m.geojson"),
                   key=lambda p: p.stat().st_mtime, reverse=True)
    return cands[0] if cands else None

grid_path = find_grid_geojson()
grid = gpd.read_file(grid_path) if grid_path else None
if grid is not None:
    grid = grid.rename(columns={"cell_id": "unit_id"})
    print(f"grid: {len(grid)} cells from {grid_path.name}")
else:
    print("grid: not generated yet (run notebook 02 with UNIT=grid first)")

def load_csv(unit: str, profile: str) -> pd.DataFrame | None:
    fp = PROC_DIR / f"{unit}_to_nearest_library_{profile}.csv"
    if not fp.exists():
        return None
    return pd.read_csv(fp, dtype={"unit_id": str})

In [ ]:
def render_static_map(merged, scale, value_col, value_unit, title, output_path,
                      edge_width: float = 0.15, draw_districts: bool = True):
    """Render a choropleth PNG. edge_width=0 for grid (clean look without grid lines)."""
    cmap = ListedColormap(scale.colors)
    bounds = [0, *scale.cuts, 1e9]
    norm = BoundaryNorm(bounds, cmap.N)

    fig, ax = plt.subplots(figsize=(12, 14), dpi=150)
    merged.plot(
        column=value_col, cmap=cmap, norm=norm,
        edgecolor="white" if edge_width > 0 else "none",
        linewidth=edge_width,
        ax=ax,
        missing_kwds={"color": "lightgray", "label": "no data"},
    )

    if draw_districts:
        districts = villages.dissolve(by="district", as_index=False)
        districts.boundary.plot(ax=ax, color="black", linewidth=0.5)

    libs_gdf.plot(ax=ax, marker="P", color="black", markersize=40,
                  edgecolor="white", linewidth=0.5)

    handles = [Patch(facecolor=c, edgecolor="white", label=f"{lab} {value_unit}")
               for c, lab in zip(scale.colors, scale.labels)]
    handles.append(Patch(facecolor="lightgray", edgecolor="white", label="無資料"))
    ax.legend(handles=handles, title=title, loc="lower left", fontsize=9)

    ax.set_title(f"台南：{title}", fontsize=14, pad=10)
    ax.set_axis_off()
    ax.set_aspect("equal")

    fig.savefig(output_path, bbox_inches="tight", dpi=150)
    plt.show()
    plt.close(fig)
    print(f"✅ Saved {output_path.name}")


GEOMS = {"village": villages, "grid": grid}
METRICS = [
    ("drive_time", "driving", DRIVING,  "time_min",     "分鐘", "到最近圖書館行車時間"),
    ("drive_dist", "driving", DISTANCE, "distance_km",  "km",   "到最近圖書館行車距離"),
    ("walk_time",  "walking", WALKING,  "time_min",     "分鐘", "到最近圖書館步行時間"),
]

# Render all (unit, metric) combos that have data
for unit, geom in GEOMS.items():
    if geom is None:
        continue
    for metric_key, profile, scale, col, unit_lbl, title in METRICS:
        df = load_csv(unit, profile)
        if df is None:
            continue
        merged = geom.merge(
            df[["unit_id", col, "nearest_library", "method"]],
            on="unit_id", how="left",
        )
        edge = 0.15 if unit == "village" else 0.0
        out = OUTPUT_DIR / f"tainan_library_{metric_key}_{unit}_static.png"
        print(f"--- {unit} / {title} ---")
        print(f"  rows={len(merged)}, range: {merged[col].min():.2f}–{merged[col].max():.2f} {unit_lbl}")
        render_static_map(merged, scale, col, unit_lbl, title, out, edge_width=edge)
        print()

In [ ]:
import folium
from folium.features import GeoJsonTooltip

def render_interactive_map(merged, scale, value_col, value_unit, title, output_path,
                           include_district_tooltip: bool):
    merged = merged.copy()
    merged["fill_color"] = merged[value_col].apply(
        lambda v: scale.color(v) if pd.notna(v) else "#cccccc"
    )
    merged["value_str"] = merged[value_col].apply(
        lambda v: f"{v:.1f}" if pd.notna(v) else "N/A"
    )

    centroid = merged.geometry.union_all().centroid
    m = folium.Map(location=[centroid.y, centroid.x], zoom_start=11, tiles="cartodbpositron")

    if include_district_tooltip:
        # village: rich tooltip with district/name
        tooltip = GeoJsonTooltip(
            fields=["village_name", "district", "nearest_library", "value_str", "method"],
            aliases=["里", "區", "最近圖書館", f"值 ({value_unit})", "計算方式"],
            sticky=True,
        )
    else:
        # grid: simpler tooltip (no district/name)
        tooltip = GeoJsonTooltip(
            fields=["unit_id", "nearest_library", "value_str", "method"],
            aliases=["Cell", "最近圖書館", f"值 ({value_unit})", "計算方式"],
            sticky=True,
        )

    folium.GeoJson(
        merged.to_json(),
        name=title,
        style_function=lambda feat: {
            "fillColor": feat["properties"]["fill_color"],
            "color": "white", "weight": 0.3, "fillOpacity": 0.75,
        },
        tooltip=tooltip,
    ).add_to(m)

    for _, lib in libs.iterrows():
        folium.Marker(
            location=[lib["lat"], lib["lon"]],
            popup=folium.Popup(f"<b>{lib['name']}</b><br>{lib['district']}<br>{lib['address']}", max_width=300),
            icon=folium.Icon(color="black", icon="book", prefix="fa"),
        ).add_to(m)

    legend = """<div style="position: fixed; bottom: 20px; left: 20px; z-index: 9999;
        background: white; padding: 10px; border: 1px solid #999;
        font-family: sans-serif; font-size: 12px;"><b>""" + title + f" ({value_unit})</b><br>"
    legend += "".join(
        f'<div><span style="display:inline-block;width:14px;height:14px;background:{c};margin-right:6px;"></span>{lab}</div>'
        for c, lab in zip(scale.colors, scale.labels)
    )
    legend += "</div>"
    m.get_root().html.add_child(folium.Element(legend))

    m.save(str(output_path))
    print(f"✅ Saved {output_path.name}")
    return m


last = None
for unit, geom in GEOMS.items():
    if geom is None:
        continue
    for metric_key, profile, scale, col, unit_lbl, title in METRICS:
        df = load_csv(unit, profile)
        if df is None:
            continue
        merged = geom.merge(
            df[["unit_id", col, "nearest_library", "method"]],
            on="unit_id", how="left",
        )
        out = OUTPUT_DIR / f"tainan_library_{metric_key}_{unit}_interactive.html"
        last = render_interactive_map(
            merged, scale, col, unit_lbl, title, out,
            include_district_tooltip=(unit == "village"),
        )

last

In [ ]:
# === Population map (village-level only) ===
from lib.colors import POPULATION

pop_path = RAW_DIR / "tainan_population.csv"
if pop_path.exists():
    pop = pd.read_csv(pop_path, dtype={"village_id": str})[
        ["village_id", "population", "households"]
    ]
    pop_merged = villages.merge(
        pop.rename(columns={"village_id": "unit_id"}),
        on="unit_id", how="left",
    )
    n_missing = pop_merged["population"].isna().sum()
    print(f"Population merged: {len(pop_merged) - n_missing}/{len(pop_merged)} villages "
          f"({n_missing} missing)")

    # Static PNG
    render_static_map(
        pop_merged, POPULATION, "population", "人",
        "村里人口（戶籍）",
        OUTPUT_DIR / "tainan_population_village_static.png",
        edge_width=0.15, draw_districts=True,
    )

    # Interactive HTML
    pop_merged_for_html = pop_merged.copy()
    # Add a household column for tooltip clarity
    pop_merged_for_html["nearest_library"] = ""  # placeholder so renderer doesn\'t crash
    pop_merged_for_html["method"] = ""
    render_interactive_map(
        pop_merged_for_html, POPULATION, "population", "人",
        "村里人口（戶籍）",
        OUTPUT_DIR / "tainan_population_village_interactive.html",
        include_district_tooltip=True,
    )
else:
    print(f"⚠️  {pop_path.name} not found — skip population map")